In [3]:
import tlc
import datasets

In [4]:
datasets_dict = datasets.load_dataset("pyronear/pyro-sdis")

In [12]:
def anns_to_3lc_bbs(anns, image_w, image_h):
    values = anns.split()
    boxes: list[list[float]] = []
    labels: list[int] = []
    for i in range(0, len(values), 5):
        labels.append(int(values[i]) - 1)  # 0-indexed labels
        cx, cy, w, h = (float(v) for v in values[i + 1 : i + 5])
        boxes.append([cx, cy, w, h])
    instances = tlc.data_types.BoundingBoxes2D(
        image_width=image_w,
        image_height=image_h,
        bboxes=boxes,
        labels=labels,
        bbox_format="cxywh",
        normalized=True,
    )
    return instances

In [ ]:
PROJECT_NAME = "hf-pyronear"
DATASET_NAME = "pyro-sdis"

bulk_data_root = "/Users/gudbrand/Data/pyro-sdis"
tlc.register_url_alias("PYRONEAR", bulk_data_root)

for split in ["val"]:
    bulk_data_location = bulk_data_root + "/" + split
    table_writer = tlc.TableWriter(
        table_name=split,
        project_name=PROJECT_NAME,
        dataset_name=DATASET_NAME,
        if_exists="overwrite",
        schema={
            "image": tlc.schemas.ImageSchema(format="jpeg", bulk_data_location=bulk_data_location),
            "bbs": tlc.schemas.BoundingBoxes2DSchema(["fire"]),
            "weight": tlc.schemas.SampleWeightSchema()
        }
    )
    ds = datasets_dict[split]
    for i in range(len(ds)):
        row = ds[i]
        image = row["image"]
        bbs = anns_to_3lc_bbs(row["annotations"], image.width, image.height)
        table_writer.add_row(
            {
                "image": image,
                "bbs": bbs,
            }
        )
    table = table_writer.finalize()
    print(f"Wrote {len(table.table_rows)} rows to table {split}")
    print(table)
    print("-" * 100)
        

In [20]:
table = table_writer.finalize()

In [21]:
table.table_rows[0]

ImmutableDict({'image': '<PYRONEAR>val/image/0000000.jpeg', 'bbs': {'x_min': 0.0, 'y_min': 0.0, 'x_max': 1280.0, 'y_max': 720.0, 'instances': [{'bbs_2d': [{'x_min': 1191.5955810546875, 'y_min': 623.0322875976562, 'x_max': 1248.3787841796875, 'y_max': 656.0174560546875}]}, {'bbs_2d': [{'x_min': 392.4952392578125, 'y_min': 385.6776123046875, 'x_max': 420.245849609375, 'y_max': 400.0252685546875}]}], 'instances_additional_data': {'label': [0, 0]}}})